# Task 1 — 검증 단위와 판정기 비교 실험 (v2)

**목적**: 스트리밍 검증에서 (1) 응답을 어떤 단위로 잘라 판정할지, (2) 무엇으로 환각을 판정할지를
실제 수치로 비교하여 이후 설계의 근거를 확보한다.

**측정 조건**
- 문맥 범위: 문장만 / 이전 1문장 포함 / 이전 2문장 포함 / 응답 전체
- 판정기: MNLI 계열 모델 2종 (원격 코드 미사용)

**측정 지표**: AUPRC(주), AUROC(보조), 문장당 추론 지연(ms)

**통과 기준(사전 정의)**: 문맥 범위에 따른 AUPRC 차이가 5%p 이상이면 동적 조절의 근거가 성립.
차이가 그보다 작으면 가장 좁은 단위로 고정하는 편이 합리적이므로 방향을 수정한다.

> **실행 전 확인**: 런타임 유형을 **GPU(T4)** 로 설정. 
> v1에서 발생한 오류는 `trust_remote_code` 모델과 transformers 버전 고정에서 비롯되었으므로,
> v2에서는 둘 다 사용하지 않는다. 코랩 기본 transformers/torch를 그대로 쓴다.

## 1. 환경 준비

In [ ]:
# transformers / torch 버전은 고정하지 않는다 (고정 시 tokenizers 소스 빌드 실패)
!pip -q install pysbd sentencepiece scikit-learn pandas matplotlib
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git

import json, time, re, random
import numpy as np, pandas as pd, torch, transformers
import pysbd
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| torch', torch.__version__, '| device', DEVICE)

## 2. 데이터 적재

평가에는 test split만 사용한다. 설계 결정을 test 성능을 보고 내리면 과적합이 발생하므로,
본 실험은 **train split의 일부를 개발용으로 떼어 수행**하고 test는 최종 평가까지 열지 않는다.

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}

for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

print('전체 응답:', len(responses))
print(pd.Series([r['split'] for r in responses]).value_counts().to_dict())
print(pd.Series([r['task_type'] for r in responses]).value_counts().to_dict())

In [ ]:
def build_reference(src, max_chars=3000):
    """task_type별로 형태가 다른 source_info를 판정기 입력용 단일 텍스트로 변환."""
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        text = f"Question: {si.get('question','')}\n{si.get('passages','')}"
    elif tt == 'Summary':
        text = si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    else:  # Data2txt — 구조화 데이터를 평문으로 직렬화
        parts = []
        for k, v in (si.items() if isinstance(si, dict) else []):
            parts.append(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}")
        text = '\n'.join(parts)
    return text[:max_chars]

# 개발용 표본: train split에서 task_type별 균등 추출
# GPU가 없거나 시간이 부족하면 30~50으로 낮출 것
N_PER_TASK = 120
dev_pool = [r for r in responses if r['split'] == 'train']
dev = []
for tt in ['QA', 'Summary', 'Data2txt']:
    cand = [r for r in dev_pool if r['task_type'] == tt]
    dev += random.sample(cand, min(N_PER_TASK, len(cand)))
print('개발 표본 응답 수:', len(dev))

## 3. 문장 분할과 레이블 변환

사전 점검에서 환각 구간의 상당수가 문장 경계를 넘나드는 것으로 확인되었다.
따라서 **겹침 비율 임계값**을 파라미터로 두고 기준에 따라 양성 비율이 어떻게 달라지는지 먼저 확인한 뒤 확정한다.
분할기는 정규식 대신 pysbd를 사용한다(약어·소수점 오분할 방지).

In [ ]:
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def split_sentences(text):
    """(start, end, sentence) 리스트 반환."""
    out = []
    for s in seg.segment(text):
        if s.sent.strip():
            out.append((s.start, s.end, text[s.start:s.end]))
    return out

def label_sentence(a, b, labels, overlap_ratio):
    """문장 [a,b)의 환각 여부. overlap_ratio=0 이면 조금이라도 겹치면 양성."""
    covered = 0
    for l in labels:
        s, e = max(a, l['start']), min(b, l['end'])
        if e > s:
            covered += e - s
    if covered == 0:
        return 0
    return 1 if covered / max(b - a, 1) >= overlap_ratio else 0

def build_sentence_records(resp_list, overlap_ratio=0.0):
    recs = []
    for r in resp_list:
        sents = split_sentences(r['response'])
        labels = r.get('labels') or []
        for i, (a, b, s) in enumerate(sents):
            recs.append({
                'resp_id': r['id'], 'source_id': r['source_id'], 'task_type': r['task_type'],
                'sent_idx': i, 'sentence': s,
                'prev1': sents[i-1][2] if i >= 1 else '',
                'prev2': ' '.join(x[2] for x in sents[max(0,i-2):i]),
                'full_response': r['response'],
                'y': label_sentence(a, b, labels, overlap_ratio),
            })
    return pd.DataFrame(recs)

# 겹침 임계값에 따른 양성 비율 변화
for ratio in [0.0, 0.3, 0.5, 0.8]:
    t = build_sentence_records(dev, ratio)
    print(f'overlap>={ratio:.1f} → 문장 {len(t):,}개, 양성 {t.y.mean()*100:.1f}%')

In [ ]:
OVERLAP_RATIO = 0.0   # 위 출력을 보고 확정할 것
df = build_sentence_records(dev, OVERLAP_RATIO)
df['reference'] = df.source_id.map(lambda sid: build_reference(sources[sid]))
print(f'문장 {len(df):,}개, 양성 {df.y.sum():,}개 ({df.y.mean()*100:.1f}%)')
df.groupby('task_type').y.agg(['size','mean']).round(3)

## 4. 문맥 범위 조건

판정 대상은 언제나 해당 문장이지만, 판정기에 함께 넣는 문맥의 범위를 달리한다.
`full`은 응답 전체를 한 번에 판정하는 조건으로, 지연이 가장 크고 정확도가 가장 높을 것으로 가정되는 상한선이다.

In [ ]:
def make_hypothesis(row, cond):
    if cond == 'sent':  return row.sentence
    if cond == 'prev1': return (row.prev1 + ' ' + row.sentence).strip()
    if cond == 'prev2': return (row.prev2 + ' ' + row.sentence).strip()
    if cond == 'full':  return row.full_response
    raise ValueError(cond)

CONDITIONS = ['sent', 'prev1', 'prev2', 'full']

## 5. 판정기

근거 문서를 전제(premise), 생성 문장을 가설(hypothesis)로 두고 함의 확률의 반대를 환각 점수로 사용한다.
두 모델 모두 학습 없이 그대로 사용하므로 이 단계에서는 과적합 우려가 없다.

> DeBERTa 계열은 환경에 따라 아키텍처 임포트 오류가 발생할 수 있으나,
> 실행 셀에서 예외 처리되어 자동으로 건너뛴다. 판정기가 하나만 남아도 실험 목적은 달성된다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class NLIJudge:
    def __init__(self, ckpt, name=None):
        self.name = name or ckpt.split('/')[-1]
        self.tok = AutoTokenizer.from_pretrained(ckpt)
        self.m = AutoModelForSequenceClassification.from_pretrained(ckpt).to(DEVICE).eval()
        self.ent_idx = [i for i, l in self.m.config.id2label.items()
                        if l.lower().startswith('entail')][0]

    @torch.no_grad()
    def score(self, pairs):
        prem = [p for p, h in pairs]; hyp = [h for p, h in pairs]
        enc = self.tok(prem, hyp, truncation=True, max_length=512,
                       padding=True, return_tensors='pt').to(DEVICE)
        prob = self.m(**enc).logits.softmax(-1)[:, self.ent_idx]
        return (1.0 - prob).cpu().numpy()   # 함의 확률이 낮을수록 환각 점수가 높다

## 6. 실험 실행

지연은 순수 추론 시간으로 측정한다. `full` 조건은 응답 전체를 1회 판정하므로
응답 내 모든 문장이 같은 점수를 공유하며, 문장당 지연은 (응답 1회 지연 ÷ 문장 수)로 환산한다.

In [ ]:
BATCH = 16

def _timed_score(judge, pairs):
    scores, elapsed = [], 0.0
    for i in range(0, len(pairs), BATCH):
        chunk = pairs[i:i+BATCH]
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        s = judge.score(chunk)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        elapsed += time.perf_counter() - t0
        scores.extend(np.asarray(s).ravel().tolist())
    return scores, elapsed

def run(judge, cond, data):
    if cond == 'full':
        uniq = data.drop_duplicates('resp_id')[['resp_id','reference','full_response']]
        pairs = list(zip(uniq.reference, uniq.full_response))
        scores, elapsed = _timed_score(judge, pairs)
        smap = dict(zip(uniq.resp_id, scores))
        y_score = data.resp_id.map(smap).values
        ms_per_call = elapsed * 1000 / len(pairs)
    else:
        hyp = data.apply(lambda r: make_hypothesis(r, cond), axis=1).tolist()
        pairs = list(zip(data.reference.tolist(), hyp))
        scores, elapsed = _timed_score(judge, pairs)
        y_score = np.array(scores)
        ms_per_call = elapsed * 1000 / len(pairs)

    y = data.y.values
    return {
        'judge': judge.name, 'condition': cond,
        'AUPRC': average_precision_score(y, y_score),
        'AUROC': roc_auc_score(y, y_score),
        'ms_per_sentence': elapsed * 1000 / len(data),
        'ms_per_call': ms_per_call,
        'n': len(data), 'pos_rate': float(y.mean()),
    }, y_score

In [ ]:
JUDGES = [
    ('FacebookAI/roberta-large-mnli', 'RoBERTa-L-MNLI'),
    ('microsoft/deberta-large-mnli',  'DeBERTa-L-MNLI'),
]

results, raw_scores = [], {}
for ckpt, nick in JUDGES:
    try:
        judge = NLIJudge(ckpt, nick)
    except Exception as e:
        print(f'{nick} 로드 실패 — 건너뜀: {type(e).__name__}: {e}')
        continue
    for cond in CONDITIONS:
        r, sc = run(judge, cond, df)
        raw_scores[(judge.name, cond)] = sc
        results.append(r)
        print(f"{r['judge']:16} {cond:6} AUPRC {r['AUPRC']:.3f}  AUROC {r['AUROC']:.3f}  "
              f"{r['ms_per_sentence']:6.1f} ms/문장")
    del judge
    if DEVICE == 'cuda': torch.cuda.empty_cache()

res = pd.DataFrame(results)
res

## 7. 통과 기준 판정과 상충 관계 시각화

In [ ]:
THRESHOLD = 0.05   # 사전 정의: 문맥 범위에 따른 AUPRC 차이 5%p

print('=== 문맥 범위에 따른 AUPRC 변동폭 ===')
for j, g in res.groupby('judge'):
    gap = g.AUPRC.max() - g.AUPRC.min()
    best = g.loc[g.AUPRC.idxmax(), 'condition']
    worst = g.loc[g.AUPRC.idxmin(), 'condition']
    verdict = '통과 — 동적 조절의 근거 성립' if gap >= THRESHOLD else '미달 — 좁은 단위 고정이 합리적'
    print(f'{j:16} 최대 {best} / 최소 {worst} / 변동폭 {gap*100:.1f}%p → {verdict}')

print('\n=== 문맥 확대에 따른 지연 증가 (sent 조건 대비 배율) ===')
for j, g in res.groupby('judge'):
    base = g[g.condition=='sent'].ms_per_sentence.values[0]
    for _, r in g.iterrows():
        print(f'{j:16} {r.condition:6} {r.ms_per_sentence:7.1f} ms/문장  ({r.ms_per_sentence/base:.2f}배)')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7,5))
for j, g in res.groupby('judge'):
    g = g.sort_values('ms_per_sentence')
    ax.plot(g.ms_per_sentence, g.AUPRC, 'o-', label=j)
    for _, r in g.iterrows():
        ax.annotate(r.condition, (r.ms_per_sentence, r.AUPRC),
                    textcoords='offset points', xytext=(6,4), fontsize=9)
ax.set_xlabel('latency per sentence (ms)')
ax.set_ylabel('AUPRC')
ax.set_title('Latency vs Detection Quality')
ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

## 8. 보조 분석 — 어떤 문장에서 문맥이 필요한가

문맥 확대의 이득이 모든 문장에 고르게 나타나는지, 일부 문장에 집중되는지 확인한다.
후자라면 위험도에 따라 문맥을 달리 적용하는 설계의 직접적인 근거가 된다.

In [ ]:
JUDGE_FOR_ANALYSIS = res.loc[res.AUPRC.idxmax(), 'judge']
s_sent  = raw_scores.get((JUDGE_FOR_ANALYSIS, 'sent'))
s_prev2 = raw_scores.get((JUDGE_FOR_ANALYSIS, 'prev2'))

if s_sent is not None and s_prev2 is not None:
    a = df.copy()
    a['score_sent'], a['score_prev2'] = s_sent, s_prev2
    a['delta'] = a.score_prev2 - a.score_sent
    # 양성 문장은 점수가 오르고 음성 문장은 내려가야 문맥이 도움이 된 것
    a['helped'] = np.where(a.y == 1, a.delta, -a.delta)

    print(f'분석 대상 판정기: {JUDGE_FOR_ANALYSIS}')
    print('문맥 확대가 도움이 된 문장 비율: %.1f%%' % ((a.helped > 0).mean() * 100))
    print('\n문장 위치별 평균 개선폭 (앞쪽 문장일수록 참조할 문맥이 적음)')
    print(a.assign(pos=a.sent_idx.clip(upper=5)).groupby('pos').helped.mean().round(4))
    print('\n과제 유형별 평균 개선폭')
    print(a.groupby('task_type').helped.mean().round(4))
    pos = a.helped.clip(lower=0)
    print('\n개선폭 상위 10%% 문장이 전체 개선량에서 차지하는 비중: %.1f%%' %
          (pos.nlargest(max(int(len(a)*0.1),1)).sum() / max(pos.sum(), 1e-9) * 100))
else:
    print('sent/prev2 점수가 모두 있어야 분석 가능')

In [ ]:
res.to_csv('task1_results.csv', index=False)
print('저장 완료: task1_results.csv')
res